# EN3160 — Assignment 3: Neural Networks

### Hiruna Kariyawasam (230318M)
### [https://github.com/HirunaK/230318M-A03-Neural-Networks](https://github.com/HirunaK/230318M-A03-Neural-Networks)

## Setup

In [1]:
import os
import copy
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", device)

PyTorch: 2.14.1+cu126
Torchvision: 0.29.1+cu126
Device: cuda


# Question 1 — Dense network with manual forward and backpropagation

The supplied Listing 1 uses CIFAR-10, flattened \(32\times32\) RGB images, and a single linear layer. Here we add a hidden layer of 100 units with sigmoid activation and use softmax cross-entropy.

For a batch:
- \(H=\sigma(XW_1+b_1)\)
- \(Z=HW_2+b_2\)
- \(L=-\frac{1}{N}\sum_i \log p_{i,y_i}\), where \(p=\mathrm{softmax}(Z)\).

For softmax cross-entropy, the gradient of logits is \(dZ=(p-Y)/N\). We then manually calculate gradients for both layers and update parameters without autograd. This keeps the forward path and backpropagation explicit, as requested.

In [ ]:
# CIFAR-10 data (matching the assignment's supplied listing)
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 128
train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=cifar_transform)
test_set = datasets.CIFAR10(root="./data", train=False, download=True, transform=cifar_transform)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2,
                          pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2,
                         pin_memory=torch.cuda.is_available())

classes_cifar = train_set.classes
Din, HIDDEN, K = 3 * 32 * 32, 100, 10
print("Classes:", classes_cifar)
print("Training samples:", len(train_set), "| Test samples:", len(test_set))

0.7%

In [ ]:
# Manual parameters. Float32 is used for the matrix operations.
# Xavier-style scaling is more suitable for a sigmoid hidden layer than the tiny
# initialization in the original single-layer example.
g = torch.Generator().manual_seed(SEED)
W1 = (torch.randn(Din, HIDDEN, generator=g) * np.sqrt(1.0 / Din)).to(device)
b1 = torch.zeros(HIDDEN, device=device)
W2 = (torch.randn(HIDDEN, K, generator=g) * np.sqrt(1.0 / HIDDEN)).to(device)
b2 = torch.zeros(K, device=device)

epochs_q1 = 10
learning_rate = 0.05
lr_decay = 0.9
train_loss_history_q1 = []
train_acc_history_q1 = []
test_acc_history_q1 = []

def evaluate_manual(loader):
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device).view(images.shape[0], -1)
            labels = labels.to(device)
            hidden = torch.sigmoid(images @ W1 + b1)
            logits = hidden @ W2 + b2
            loss = F.cross_entropy(logits, labels, reduction="sum")
            loss_sum += loss.item()
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, 100.0 * correct / total

for epoch in range(epochs_q1):
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        X = images.to(device).view(images.shape[0], -1)
        y = labels.to(device)
        N = X.shape[0]

        # Forward pass
        A1 = X @ W1 + b1
        H = torch.sigmoid(A1)
        logits = H @ W2 + b2
        loss = F.cross_entropy(logits, y)

        # Manual backpropagation (mean cross-entropy)
        probs = torch.softmax(logits, dim=1)
        probs[torch.arange(N, device=device), y] -= 1.0
        dZ2 = probs / N
        dW2 = H.T @ dZ2
        db2 = dZ2.sum(dim=0)

        dH = dZ2 @ W2.T
        dA1 = dH * H * (1.0 - H)  # derivative of sigmoid
        dW1 = X.T @ dA1
        db1 = dA1.sum(dim=0)

        # Parameter update; no autograd or optimizer is used
        with torch.no_grad():
            W1 -= learning_rate * dW1
            b1 -= learning_rate * db1
            W2 -= learning_rate * dW2
            b2 -= learning_rate * db2

        running_loss += loss.item() * N
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += N

    train_loss_history_q1.append(running_loss / total)
    train_acc_history_q1.append(100.0 * correct / total)
    _, test_acc = evaluate_manual(test_loader)
    test_acc_history_q1.append(test_acc)
    print(f"Epoch {epoch+1:02d}/{epochs_q1} | loss={train_loss_history_q1[-1]:.4f} "
          f"| train acc={train_acc_history_q1[-1]:.2f}% | test acc={test_acc:.2f}%")
    learning_rate *= lr_decay

final_train_loss_q1, final_train_acc_q1 = evaluate_manual(train_loader)
final_test_loss_q1, final_test_acc_q1 = evaluate_manual(test_loader)
print(f"\nFinal training accuracy: {final_train_acc_q1:.2f}%")
print(f"Final test accuracy:     {final_test_acc_q1:.2f}%")

In [ ]:
epochs = np.arange(1, epochs_q1 + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, train_loss_history_q1, marker="o")
ax[0].set(title="Q1: Training loss", xlabel="Epoch", ylabel="Cross-entropy loss")
ax[0].grid(True)
ax[1].plot(epochs, train_acc_history_q1, marker="o", label="Training")
ax[1].plot(epochs, test_acc_history_q1, marker="o", label="Test")
ax[1].set(title="Q1: Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
ax[1].legend()
ax[1].grid(True)
plt.tight_layout()
plt.show()

### Question 1 — Results and discussion

The final training and test accuracies are printed above and plotted by epoch. Compare the two values:
- A large gap between training and test accuracy can indicate overfitting.
- Low accuracy on both sets can indicate underfitting, insufficient optimization, or that a fully connected model is not exploiting image structure effectively.
- This model uses a manually coded gradient calculation, so it is simpler to inspect but more error-prone than PyTorch autograd.

# Question 2 — MNIST: LeNet-5 versus a modern small CNN

Both networks are trained on MNIST for 10 epochs with the same train/test split, batch size, and optimizer settings. Images are normalized to mean 0.1307 and standard deviation 0.3081. The models output ten logits; `CrossEntropyLoss` internally applies the log-softmax operation, so a separate softmax layer should not be added to the model.

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
mnist_train = datasets.MNIST("./data", train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST("./data", train=False, download=True, transform=mnist_transform)

mnist_batch_size = 128
mnist_train_loader = DataLoader(mnist_train, batch_size=mnist_batch_size, shuffle=True,
                                num_workers=2, pin_memory=torch.cuda.is_available())
mnist_test_loader = DataLoader(mnist_test, batch_size=mnist_batch_size, shuffle=False,
                               num_workers=2, pin_memory=torch.cuda.is_available())

## 2(a) LeNet-5

This is a LeNet-5-style architecture adapted for the \(28\times28\) MNIST input: convolution, average pooling, two further fully connected stages, and ten output logits. The first convolution/pooling stages produce a \(4\times4\times16\) feature map, which is flattened before the classifier.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)       # 28 -> 24
        self.pool1 = nn.AvgPool2d(2)                      # 24 -> 12
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)      # 12 -> 8
        self.pool2 = nn.AvgPool2d(2)                      # 8 -> 4
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)
        x = torch.tanh(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.fc3(x)

## 2(b) Modern small CNN

The second model has three convolutional layers, batch normalization, ReLU activations, max pooling, and dropout. The architecture is intentionally compact while including common modern CNN training components.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28 -> 14

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 14 -> 7

            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

def train_mnist_model(model, name, epochs=10, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    start = time.time()
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, mnist_train_loader, criterion, optimizer)
        te_loss, te_acc = run_epoch(model, mnist_test_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        print(f"{name} | Epoch {epoch+1:02d}/{epochs} | "
              f"train loss {tr_loss:.4f}, acc {tr_acc:.2f}% | "
              f"test loss {te_loss:.4f}, acc {te_acc:.2f}%")
    print(f"{name} training time: {(time.time()-start)/60:.1f} min")
    return model, history

lenet, lenet_history = train_mnist_model(LeNet5(), "LeNet-5", epochs=10)
small_cnn, cnn_history = train_mnist_model(SmallCNN(), "Small CNN", epochs=10)

In [ ]:
def plot_history(histories):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for name, h in histories.items():
        e = np.arange(1, len(h["train_loss"]) + 1)
        axes[0, 0].plot(e, h["train_loss"], marker=".", label=name)
        axes[0, 1].plot(e, h["test_loss"], marker=".", label=name)
        axes[1, 0].plot(e, h["train_acc"], marker=".", label=name)
        axes[1, 1].plot(e, h["test_acc"], marker=".", label=name)
    axes[0, 0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
    axes[0, 1].set(title="Test loss", xlabel="Epoch", ylabel="Cross-entropy")
    axes[1, 0].set(title="Training accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    axes[1, 1].set(title="Test accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    for a in axes.flat:
        a.grid(True)
        a.legend()
    plt.tight_layout()
    plt.show()

plot_history({"LeNet-5": lenet_history, "Small CNN": cnn_history})

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            y_pred.extend(logits.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())
    return np.asarray(y_true), np.asarray(y_pred)

def show_confusion(model, title):
    y_true, y_pred = collect_predictions(model, mnist_test_loader)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(10)))
    disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    return y_true, y_pred, cm

lenet_true, lenet_pred, lenet_cm = show_confusion(lenet, "LeNet-5 — MNIST test confusion matrix")
cnn_true, cnn_pred, cnn_cm = show_confusion(small_cnn, "Small CNN — MNIST test confusion matrix")

In [ ]:
# Final comparison table
import pandas as pd
comparison_q2 = pd.DataFrame([
    {
        "Model": "LeNet-5",
        "Final train accuracy (%)": lenet_history["train_acc"][-1],
        "Final test accuracy (%)": lenet_history["test_acc"][-1],
        "Final train loss": lenet_history["train_loss"][-1],
        "Final test loss": lenet_history["test_loss"][-1],
        "Parameters": sum(p.numel() for p in lenet.parameters())
    },
    {
        "Model": "Small CNN",
        "Final train accuracy (%)": cnn_history["train_acc"][-1],
        "Final test accuracy (%)": cnn_history["test_acc"][-1],
        "Final train loss": cnn_history["train_loss"][-1],
        "Final test loss": cnn_history["test_loss"][-1],
        "Parameters": sum(p.numel() for p in small_cnn.parameters())
    }
])
comparison_q2.round(4)

### Question 2 — Results and discussion

Use the printed table, plots, and confusion matrices to compare the models. The confusion matrix shows which digit classes are being confused. The modern CNN may benefit from batch normalization, ReLU, and dropout, but the actual result depends on initialization, training time, and the runtime environment. Discuss the measured results rather than assuming one model must win.

# Question 3 — Transfer learning with pretrained ResNet18

The assignment asks for the Hymenoptera dataset (ants and bees) and pretrained ImageNet-1K ResNet18, using:
1. **Fine-tuning:** update all network parameters.
2. **Feature extraction:** freeze the pretrained backbone and train only the final classification layer.

The dataset is downloaded using `torchvision.datasets.ImageFolder` from the standard PyTorch transfer-learning tutorial URL. ResNet18 expects three-channel images; the preprocessing below resizes/crops images to \(224\times224\) and uses ImageNet normalization.

In [ ]:
from urllib.request import urlretrieve
import zipfile

data_root = "./data"
hymenoptera_dir = os.path.join(data_root, "hymenoptera_data")
if not os.path.isdir(hymenoptera_dir):
    os.makedirs(data_root, exist_ok=True)
    zip_path = os.path.join(data_root, "hymenoptera_data.zip")
    urlretrieve(
        "https://download.pytorch.org/tutorial/hymenoptera_data.zip",
        zip_path
    )
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_root)

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(hymenoptera_dir, x), data_transforms[x])
    for x in ["train", "val"]
}
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=4, shuffle=(x == "train"),
                  num_workers=2, pin_memory=torch.cuda.is_available())
    for x in ["train", "val"]
}
dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}
class_names = image_datasets["train"].classes
print("Classes:", class_names)
print("Dataset sizes:", dataset_sizes)

## Shared training and evaluation functions

In [ ]:
def train_transfer_model(model, criterion, optimizer, scheduler=None, num_epochs=10):
    since = time.time()
    best_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 30)
        for phase in ["train", "val"]:
            model.train(phase == "train")
            running_loss, running_corrects = 0.0, 0

            context = torch.enable_grad() if phase == "train" else torch.no_grad()
            with context:
                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(device), labels.to(device)
                    if phase == "train":
                        optimizer.zero_grad(set_to_none=True)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels).item()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(100.0 * epoch_acc)
            print(f"{phase:5s} loss: {epoch_loss:.4f} acc: {100*epoch_acc:.2f}%")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())

        if scheduler is not None:
            scheduler.step()
        print()

    elapsed = time.time() - since
    print(f"Training complete in {elapsed//60:.0f}m {elapsed%60:.0f}s")
    print(f"Best validation accuracy: {100*best_acc:.2f}%")
    model.load_state_dict(best_weights)
    return model, history, best_acc

def plot_transfer_history(history, title):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    ax[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
    ax[0].set(title=f"{title}: loss", xlabel="Epoch", ylabel="Cross-entropy")
    ax[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    ax[1].plot(epochs, history["val_acc"], marker="o", label="Validation")
    ax[1].set(title=f"{title}: accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    for a in ax:
        a.grid(True)
        a.legend()
    plt.tight_layout()
    plt.show()

def get_resnet18_pretrained():
    # New torchvision API with fallback for older installations
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
    except AttributeError:
        model = models.resnet18(pretrained=True)
    return model

## 3(a) Fine-tuning

All ResNet18 parameters are trainable. The final fully connected layer is replaced with a two-class classifier. A small learning rate is used because the backbone is already pretrained.

In [ ]:
resnet_finetune = get_resnet18_pretrained()
num_features = resnet_finetune.fc.in_features
resnet_finetune.fc = nn.Linear(num_features, len(class_names))
resnet_finetune = resnet_finetune.to(device)

criterion_ft = nn.CrossEntropyLoss()
optimizer_ft = optim.SGD(resnet_finetune.parameters(), lr=1e-3, momentum=0.9)
scheduler_ft = optim.lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

resnet_finetune, history_ft, best_acc_ft = train_transfer_model(
    resnet_finetune, criterion_ft, optimizer_ft, scheduler_ft, num_epochs=10
)
plot_transfer_history(history_ft, "ResNet18 fine-tuning")

## 3(b) Feature extraction

All pretrained ResNet18 parameters are frozen. Only the replacement final fully connected layer is trainable. This is typically faster and requires fewer trainable parameters than fine-tuning.

In [ ]:
resnet_features = get_resnet18_pretrained()
for param in resnet_features.parameters():
    param.requires_grad = False

num_features = resnet_features.fc.in_features
resnet_features.fc = nn.Linear(num_features, len(class_names))
resnet_features = resnet_features.to(device)

trainable_params = [p for p in resnet_features.parameters() if p.requires_grad]
print("Trainable parameters:", sum(p.numel() for p in trainable_params))
print("Total parameters:", sum(p.numel() for p in resnet_features.parameters()))

criterion_fe = nn.CrossEntropyLoss()
optimizer_fe = optim.SGD(trainable_params, lr=1e-3, momentum=0.9)
scheduler_fe = optim.lr_scheduler.StepLR(optimizer_fe, step_size=7, gamma=0.1)

resnet_features, history_fe, best_acc_fe = train_transfer_model(
    resnet_features, criterion_fe, optimizer_fe, scheduler_fe, num_epochs=10
)
plot_transfer_history(history_fe, "ResNet18 feature extraction")

In [ ]:
# Summary comparison for Question 3
comparison_q3 = pd.DataFrame([
    {
        "Method": "Fine-tuning",
        "Best validation accuracy (%)": 100 * best_acc_ft,
        "Trainable parameters": sum(p.numel() for p in resnet_finetune.parameters() if p.requires_grad),
        "Total parameters": sum(p.numel() for p in resnet_finetune.parameters())
    },
    {
        "Method": "Feature extraction",
        "Best validation accuracy (%)": 100 * best_acc_fe,
        "Trainable parameters": sum(p.numel() for p in resnet_features.parameters() if p.requires_grad),
        "Total parameters": sum(p.numel() for p in resnet_features.parameters())
    }
])
comparison_q3.round(2)

### Question 3 — Results and discussion

Report the best validation accuracy for each method from the comparison table. Fine-tuning can adapt both the pretrained features and classifier to the ants/bees domain, but it updates many more parameters. Feature extraction keeps the pretrained backbone fixed and trains only the classifier, so it is computationally cheaper and can be effective when the dataset is small. Use the observed validation results to decide which performed better in this run.

**Important:** validation accuracy is used here because the standard Hymenoptera tutorial dataset provides `train` and `val` folders rather than a separate official test folder.

# Overall conclusion

Fill in the conclusions below after running all sections. Include the measured results from this notebook in your submitted report.

- **Question 1:** Training accuracy = ______%; test accuracy = ______%.
- **Question 2:** LeNet-5 test accuracy = ______%; Small CNN test accuracy = ______%. Explain the difference using the learning curves and confusion matrices.
- **Question 3:** Fine-tuning best validation accuracy = ______%; feature extraction best validation accuracy = ______%. Compare accuracy, training cost, and number of trainable parameters.